# 📝 Notebook 5: Content-Based Filtering

**Mục tiêu:**
- Gợi phim dựa trên **nội dung phim** (genres), không cần user data
- TF-IDF vectorization trên genres
- Cosine similarity để tìm phim tương tự

**Khác gì Collaborative Filtering?**
- CF: "Người khác thích gì → bạn cũng thích"
- Content-Based: "Phim này có gì → tìm phim tương tự"

## 1. Lý thuyết: Content-Based Filtering

```
Ý tưởng: Mỗi phim có đặc trưng riêng (genres)
          → Tìm phim có đặc trưng GIỐNG phim user đã thích

Các bước:
  1. Trích xuất features từ phim (genres)
  2. Chuyển thành vector (TF-IDF)
  3. Tính similarity giữa các vectors (Cosine)
  4. Với mỗi phim user thích → tìm K phim giống nhất

Ưu điểm:
  ✓ Không cần data từ user khác
  ✓ Giải quyết cold-start cho items mới
  ✓ Gợi explained được ("vì bạn thích X")

Nhược điểm:
  ✗ Chỉ gợi phim "tương tự" → thiếu diversity
  ✗ Cần features tốt (genres, description, actors...)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

ratings = pd.read_csv('data/processed/ratings_clean.csv')
movies  = pd.read_csv('data/processed/movies_clean.csv')

print(f'✅ Loaded: {len(movies):,} movies')

## 2. Chuẩn bị Genres

In [ ]:
# Xem ví dụ genres
print('Ví dụ genres:')
for title, genres in movies[['title','genres']].head(10).values:
    print(f'  {title[:50]:<52} | {genres}')

In [ ]:
# Chuyển genres: "Action|Comedy|Drama" → "Action Comedy Drama"
movies['genres_clean'] = movies['genres'].str.replace('|', ' ', regex=False)

# Xem lại
print('Genres sau khi clean:')
movies[['title','genres','genres_clean']].head(5)

## 3. TF-IDF Vectorization

TF-IDF đo mức độ quan trọng của từ trong document:
- **TF** (Term Frequency): tần suất xuất hiện
- **IDF** (Inverse Document Frequency): giảm trọng số từ phổ biến

In [ ]:
# TF-IDF trên genres
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['genres_clean'])

print(f'Matrix shape: {tfidf_matrix.shape}')
print(f'  → {tfidf_matrix.shape[0]} movies × {tfidf_matrix.shape[1]} features (genres/terms)')

# Xem features (từ trong vocabulary)
print(f'\nVocabulary: {tfidf.get_feature_names_out()}')

In [ ]:
# Ví dụ: vector của phim đầu tiên
sample_idx = 0
sample_vec = tfidf_matrix[sample_idx].toarray().flatten()
non_zero = [(tfidf.get_feature_names_out()[i], sample_vec[i])
            for i in range(len(sample_vec)) if sample_vec[i] > 0]

print(f'Phim: {movies.iloc[sample_idx]["title"]}')
print(f'Genres: {movies.iloc[sample_idx]["genres"]}')
print(f'\nTF-IDF vector (sparse non-zero values):')
for term, weight in non_zero:
    print(f'  {term}: {weight:.4f}')

## 4. Cosine Similarity

Tính similarity giữa tất cả các cặp phim:

In [ ]:
# Tính cosine similarity giữa tất cả phim
cosine_sim = cosine_similarity(tfidf_matrix)

print(f'Similarity matrix: {cosine_sim.shape}')
print(f'  → Ma trận đối xứng {cosine_sim.shape[0]}×{cosine_sim.shape[0]}')
print(f'\nSample similarity scores:')
for i in range(5):
    print(f'  Movie[0] vs Movie[{i}]: {cosine_sim[0][i]:.4f}')

In [ ]:
# Vẽ heatmap (sample 50 phim đầu)
sample_size = 50
fig, ax = plt.subplots(figsize=(10, 8))

sns = __import__('seaborn')
import seaborn as sns

sns.heatmap(cosine_sim[:sample_size, :sample_size],
            cmap='Blues', ax=ax, square=True,
            xticklabels=False, yticklabels=False)
ax.set_title(f'Cosine Similarity Matrix (50 phim đầu tiên)')
ax.set_xlabel('Movies')
ax.set_ylabel('Movies')

plt.tight_layout()
plt.savefig('results/charts/05_content_similarity_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Tìm phim tương tự

In [ ]:
# Tạo mapping: movieId → index trong dataframe
movie_idx = pd.Series(movies.index, index=movies['movieId'])


def find_similar_movies(movie_id, cosine_sim, movies_df, movie_idx, top_n=10):
    """Tìm N phim tương tự nhất với movie_id."""
    idx = movie_idx[movie_id]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n+1]  # Bỏ chính nó

    results = []
    for i, score in sim_scores:
        results.append({
            'movieId': movies_df.iloc[i]['movieId'],
            'title': movies_df.iloc[i]['title'],
            'genres': movies_df.iloc[i]['genres'],
            'similarity': round(score, 4)
        })
    return results


# Demo: Tìm phim tương tự "Toy Story (1995)"
toy_story = movies[movies['title'].str.startswith('Toy Story')].iloc[0]
print(f'Phim gốc: {toy_story["title"]} | {toy_story["genres"]}')
print('\nPhim tương tự:')
pd.DataFrame(find_similar_movies(toy_story['movieId'], cosine_sim, movies, movie_idx, top_n=5))

## 6. Gợi ý cho User cụ thể

In [ ]:
def recommend_content(user_id, ratings_df, movies_df, cosine_sim, movie_idx, top_n=10):
    """Gợi phim cho user bằng Content-Based."""
    # Lấy phim user đã đánh giá cao nhất
    user_ratings = ratings_df[ratings_df['userId'] == user_id]
    top_rated = user_ratings.nlargest(5, 'rating')

    scores = {}
    for _, row in top_rated.iterrows():
        mid = row['movieId']
        if mid in movie_idx.index:
            idx = movie_idx[mid]
            sims = cosine_sim[idx]
            for i, s in enumerate(sims):
                # Cộng dồn score (đánh giá cao → weight lớn)
                if i not in user_ratings['movieId'].values:
                    scores[i] = scores.get(i, 0) + s * row['rating']

    sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    results = []
    for idx, score in sorted_scores[:top_n]:
        results.append({
            'movieId': movies_df.iloc[idx]['movieId'],
            'title': movies_df.iloc[idx]['title'],
            'genres': movies_df.iloc[idx]['genres'],
            'score': round(score, 2)
        })
    return results


user_id = 1
recs = recommend_content(user_id, ratings, movies, cosine_sim, movie_idx, top_n=10)
print(f'🎬 Top 10 gợi ý cho User {user_id} (Content-Based):')
pd.DataFrame(recs)

## 7. Đánh giá Content-Based

In [ ]:
# Đánh giá bằng RMSE (so với actual ratings)
from surprise import Dataset, Reader

# Build full model
model = {
    'cosine_sim': cosine_sim,
    'movie_idx': movie_idx,
    'movies': movies
}

# Tính predicted ratings
user_id_test = 1
user_ratings = ratings[ratings['userId'] == user_id_test].copy()

user_recs = recommend_content(user_id_test, ratings, movies, cosine_sim, movie_idx, top_n=50)
rec_movie_ids = [r['movieId'] for r in user_recs]

actual_vs_pred = []
for _, row in user_ratings.iterrows():
    if row['movieId'] in rec_movie_ids:
        pred_score = [r['score'] for r in user_recs if r['movieId'] == row['movieId']][0]
        # Normalize score về 1-5
        max_score = max(r['score'] for r in user_recs)
        pred_rating = 1 + 4 * (pred_score / max_score) if max_score > 0 else 3
        actual_vs_pred.append({
            'movieId': row['movieId'],
            'actual': row['rating'],
            'predicted': round(pred_rating, 2)
        })

df_eval = pd.DataFrame(actual_vs_pred)
if len(df_eval) > 0:
    mae = (df_eval['actual'] - df_eval['predicted']).abs().mean()
    print(f'Content-Based MAE (User {user_id_test}): {mae:.4f}')
    print('\nSample predictions:')
    print(df_eval.head(10))

## 8. Tổng kết

**Content-Based vs Collaborative Filtering:**

| Tiêu chí | Content-Based | Collaborative Filtering |
|----------|--------------|--------------------------|
| Dữ liệu | Genres, features phim | Hành vi user khác |
| Cold-start item | ✅ Giải quyết được | ❌ Khó khăn |
| Gợi đa dạng | ❌ Thiên về tương tự | ✅ Khám phá mới |
| Explained | ✅ "Vì bạn thích X" | ❌ Không rõ lý do |

**Kết luận:** Content-Based hữu ích khi cần gợi phim mới (cold-start) hoặc khi không có nhiều user data. Tuy nhiên, để đạt chất lượng cao nhất, nên **kết hợp với CF** → Hybrid.